Solucion al reto de la clasificacion de resenias de Mex-Rest

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
from datasets import Dataset, load_dataset, DatasetDict
import pandas as pd
#Lectura del archivo donde se encuentran los datos de entrenamiento
data = pd.read_excel('/content/drive/MyDrive/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx')
data_test = pd.read_excel('/content/drive/MyDrive/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx')
#Unicamente utilizamos el texto y las demás columnas se eliminan
data = data.drop(columns=['Town','Region','Type'])
data_test = data_test.drop(columns=['Town','Region','Type'])
print(data.head(5),"\n")

from datasets import Dataset, DatasetDict
data = data.rename(columns={'Polarity': 'labels'})#para estandarizar con los modelos de transfomers
data['labels'] = data['labels'].astype(int)#para que las clases estén de en el rango de [0-4]
data['labels'] = data['labels'] - 1#para que las clases estén de en el rango de [0-4]
print(data.head(5),"\n")

# 2. Crear dataset desde pandas
ds_train = Dataset.from_pandas(data)
ds_test = Dataset.from_pandas(data_test)


# 6. Definir diccionarios de mapeo
# diccionarios labels
id2label = {0: "very negative", 1: "negative", 2: "neutral", 3: "positive", 4: "very positive"}
label2id = {"very negative": 0, "negative": 1, "neutral": 2, "positive": 3, "very positive": 4}

#Se convierte a un datasetDict
ds_train = Dataset.from_pandas(data)
ds_dict = {'train' : ds_train}
dataset = DatasetDict(ds_dict)
dataset

                                              Review  Polarity
0  Un Restaurante te invita por su ambiente tan a...       2.0
1  Pagamos 25 pesos por la entrada y no es gran c...       3.0
2  Mi esposa y yo nos alojamos en el Dreams por 4...       3.0
3  La única decepción puede no ser José Cuervo, p...       2.0
4  Cuando leí los comentarios sobre cómo son las ...       1.0 

                                              Review  labels
0  Un Restaurante te invita por su ambiente tan a...       1
1  Pagamos 25 pesos por la entrada y no es gran c...       2
2  Mi esposa y yo nos alojamos en el Dreams por 4...       2
3  La única decepción puede no ser José Cuervo, p...       1
4  Cuando leí los comentarios sobre cómo son las ...       0 



DatasetDict({
    train: Dataset({
        features: ['Review', 'labels'],
        num_rows: 5000
    })
})

In [27]:
data.info() # no tenemos nulos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  5000 non-null   object
 1   labels  5000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 78.3+ KB


In [28]:
# 1 2
model_checkpoint = "finiteautomata/beto-sentiment-analysis"
# 3
#model_checkpoint = "tabularisai/multilingual-sentiment-analysis"
# 4
#model_checkpoint = "cardiffnlp/twitter-roberta-base-sentiment-latest"
# 5 no funciono
#model_checkpoint = "finiteautomata/bertweet-base-sentiment-analysis"



In [29]:
from transformers import AutoTokenizer
from transformers import BertForSequenceClassification, BertTokenizerFast, DistilBertTokenizerFast, RobertaTokenizerFast,BertweetTokenizer

tokenizer = BertTokenizerFast.from_pretrained(model_checkpoint)
#tokenizer = DistilBertTokenizerFast.from_pretrained(model_checkpoint)
#tokenizer = RobertaTokenizerFast.from_pretrained(model_checkpoint)
#tokenizer = BertweetTokenizer.from_pretrained(model_checkpoint)

In [30]:
def tokenize_fn(examples):
    return tokenizer(
        examples["Review"],
        padding="max_length",  # o True si usas DataCollator
        truncation=True,
        max_length=128
    )

In [31]:
tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["Review"])
tokenized_dataset


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
})

In [32]:
tokenized_dataset["train"][1]

{'labels': 2,
 'input_ids': [4,
  1895,
  3557,
  2329,
  16038,
  1096,
  1030,
  4336,
  1042,
  1084,
  1058,
  1523,
  2467,
  1009,
  1378,
  2956,
  2536,
  1058,
  1038,
  1084,
  19021,
  2635,
  1017,
  1355,
  1423,
  4935,
  1789,
  2437,
  27979,
  1110,
  13406,
  1732,
  1036,
  1030,
  21646,
  1038,
  1404,
  17305,
  30934,
  1009,
  5,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,


In [33]:
from transformers import BertForSequenceClassification, BertTokenizerFast, DistilBertForSequenceClassification, RobertaForSequenceClassification


model =BertForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at finiteautomata/beto-sentiment-analysis and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([5, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([5]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
print(len(id2label))  # Should show your label mapping
print(len(label2id))  # Inverse mapping

5
5


In [35]:
print(model.classifier)  # Should show Linear(in_features=768, out_features=5)

Linear(in_features=768, out_features=5, bias=True)


In [36]:
from transformers import DataCollatorWithPadding
import numpy as np
from datasets import load_metric

# Se encarga del padding dinámico
data_collator = DataCollatorWithPadding(tokenizer) #pad_to_multiple_of=8

accuracy_metric = load_metric("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


In [37]:
import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="bert-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    load_best_model_at_end=True,
    remove_unused_columns=False,
    metric_for_best_model="accuracy",
)

#training_args = TrainingArguments(
 #   output_dir="beto-optimized",
 #   eval_strategy="epoch",
 #   save_strategy="epoch",
 #   learning_rate=3e-5,  # Increased from 2e-5
 #   per_device_train_batch_size=32 if torch.cuda.is_available() else 8,
 #   per_device_eval_batch_size=64,  # Larger eval batches
 #   num_train_epochs=3,  # Increased epochs
 #   warmup_ratio=0.1,  # Critical for Beto
 #   weight_decay=0.015,  # Adjusted regularization
 #   lr_scheduler_type="cosine_with_restarts",  # Better learning curve
 #   gradient_accumulation_steps=2,  # Simulates larger batch size
 #   fp16=True,  # Enable mixed-precision
 #   logging_steps=100,
 #   load_best_model_at_end=True,
 #   metric_for_best_model="accuracy",  # Better for imbalanced data than accuracy
 #   greater_is_better=True,
 #   report_to="wandb"  # For tracking experiments
#)

In [38]:
# Split your dataset (e.g., 80% train, 10% validation, 10% test)
split_dataset = tokenized_dataset["train"].train_test_split(
    test_size=0.2, #0.2
    shuffle=True,
    seed=42
)

# Then split the test portion further into validation and test
final_splits = split_dataset["test"].train_test_split(test_size=0.5)

In [39]:
print(split_dataset.keys())  # Will show available splits (e.g., ['train', 'validation'])

dict_keys(['train', 'test'])


In [40]:
print(final_splits.keys())  # Will show available splits (e.g., ['train', 'validation'])

dict_keys(['train', 'test'])


In [41]:
print(tokenized_dataset["train"]["labels"][:5])  # Check first 5 labels

[1, 2, 2, 1, 0]


In [42]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=final_splits["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)#tokenizer=tokenizer

trainer.train()



Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.020669,0.544000
2,0.992300,0.974521,0.566000
3,0.992300,1.030201,0.568000


TrainOutput(global_step=750, training_loss=0.8786775919596355, metrics={'train_runtime': 190.4102, 'train_samples_per_second': 63.022, 'train_steps_per_second': 3.939, 'total_flos': 789354427392000.0, 'train_loss': 0.8786775919596355, 'epoch': 3.0})

In [43]:
print(f"Model device: {next(model.parameters()).device}")
print(f"Label range: {split_dataset['train'].unique('labels')}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size} vs Model embeddings: {model.config.vocab_size}")

Model device: cuda:0


Flattening the indices:   0%|          | 0/4000 [00:00<?, ? examples/s]

Label range: [2, 3, 0, 4, 1]
Tokenizer vocab size: 31002 vs Model embeddings: 31006


In [44]:
predictions = trainer.predict(final_splits["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print("Test Accuracy:", accuracy_metric.compute(predictions=y_pred, references=y_true))



Test Accuracy: {'accuracy': 0.568}


In [21]:
from transformers import pipeline

clf_pipe = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0  # cambia a -1 si usas CPU
)


Device set to use cuda:0


In [22]:
single = clf_pipe("Este es un texto de prueba.", truncation=True)
print(single)

[{'label': 'positive', 'score': 0.42497503757476807}]


In [23]:
frases_test = [
    # Positivas
    "Me encantó el servicio, todo fue perfecto.",
    "La comida estaba deliciosa y el lugar muy limpio.",
    "Una experiencia excelente, sin duda volveré.",

    # Neutrales
    "El restaurante está ubicado cerca del centro.",
    "Fuimos un martes por la tarde, no había mucha gente.",
    "El menú incluye opciones para vegetarianos.",

    # Negativas
    "La atención fue lenta y desorganizada.",
    "El lugar estaba sucio y ruidoso.",
    "No volvería, la comida no valió la pena."
]


batch = clf_pipe(frases_test, batch_size=8, truncation=True)
for res in batch:
    print(res)


{'label': 'very positive', 'score': 0.9128063321113586}
{'label': 'very positive', 'score': 0.614967942237854}
{'label': 'very positive', 'score': 0.917231023311615}
{'label': 'positive', 'score': 0.634830892086029}
{'label': 'neutral', 'score': 0.47795242071151733}
{'label': 'positive', 'score': 0.6024758219718933}
{'label': 'negative', 'score': 0.4329051375389099}
{'label': 'very negative', 'score': 0.5612084865570068}
{'label': 'very negative', 'score': 0.6842820644378662}


In [24]:
clf_pipe = pipeline(
    "text-classification",
    model=trainer.model,#Este modelo y tokenizador deben ser los que tú entrenaste
    tokenizer=tokenizer,
    device=0  # cambia a -1 si usas CPU
)

# Roberta
#clf_pipe = pipeline(
  #  "text-classification",
 #   model=trainer.model,
  #  tokenizer=tokenizer,
  #  device=0,  # or -1 for CPU
  #  truncation=True,
   # padding="max_length",  # Changed from True to "max_length"
   # max_length=512  # Explicitly set max length
#)


frases_test2 = data_test['Review'].tolist();
batch = clf_pipe(frases_test2, padding = True, truncation=True)


Device set to use cuda:0


In [25]:
escribir = open('/content/drive/MyDrive/Datos-MeIA-Reto-01/Alpha-Run.txt', 'w')#recuerda poner la dirección qué tu quieras
#ejemplo CIMAT-3.txt
i = 0;
for res in batch:
    escribir.write(f"MeIA {i} {label2id[res['label']] + 1}\n")
    i+=1;
escribir.close()